# TF-IDF Class Activity

Run cells top to bottom. Kernel working directory should be `notebooks/`.

In [1]:
# Setup: make sure the repo root is on sys.path so we can import src/ and data/
import sys
import os

# Works whether the kernel starts in the repo root or in notebooks/
_cwd = os.getcwd()
REPO_ROOT = _cwd if os.path.isdir(os.path.join(_cwd, "src")) else os.path.abspath(os.path.join(_cwd, ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.load_data import load_documents

titles, documents = load_documents()
print(f"{len(documents)} documents loaded.")
for t, d in zip(titles, documents):
    print(f"  - {t}: {len(d.split())} words")

10 documents loaded.
  - AI in Healthcare: 85 words
  - Climate Change: 83 words
  - Basketball Strategy: 85 words
  - Italian Pasta: 91 words
  - Space Exploration: 86 words
  - Cybersecurity: 88 words
  - Music Theory: 87 words
  - Gardening Vegetables: 89 words
  - Economics Inflation: 89 words
  - Public Health: 91 words


## Step 1 — Build the TF-IDF matrix

TODO: import `TfidfVectorizer` and fit it on `documents`.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)
print("Matrix shape:", X.shape)              # (10, n_unique_terms)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Matrix shape: (10, 537)
Vocabulary size: 537


## Step 2 — Inspect the matrix as a DataFrame

TODO: convert to a pandas DataFrame and print the top-5 TF-IDF terms per document.

In [3]:
import pandas as pd

df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=titles,
)

print("\nSample of the TF-IDF matrix (first 3 docs × 10 terms):")
print(df.iloc[:3, :10].round(3))

print("\nTop 5 TF-IDF terms per document:")
for t in titles:
    top5 = df.loc[t].sort_values(ascending=False).head(5)
    print(f"\n{t}:")
    for term, score in top5.items():
        print(f"  {term:<20s} {score:.3f}")


Sample of the TF-IDF matrix (first 3 docs × 10 terms):
                     ability  accelerate  access  accountability  accuracy  \
AI in Healthcare        0.00         0.0     0.0           0.125     0.125   
Climate Change          0.00         0.0     0.0           0.000     0.000   
Basketball Strategy     0.13         0.0     0.0           0.000     0.000   

                     accurate  actions  activity  adaptation  add  
AI in Healthcare        0.000    0.000       0.0       0.000  0.0  
Climate Change          0.125    0.125       0.0       0.125  0.0  
Basketball Strategy     0.000    0.000       0.0       0.000  0.0  

Top 5 TF-IDF terms per document:

AI in Healthcare:
  ai                   0.374
  healthcare           0.249
  use                  0.165
  waiting              0.125
  transforming         0.125

Climate Change:
  change               0.250
  emissions            0.250
  energy               0.212
  climate              0.212
  burning              0.125

## Step 3 — Query similarity challenge

TODO: rank documents by cosine similarity for a given query.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity


def rank_documents(query: str):
    """Print documents ranked by cosine similarity to the query."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, X).flatten()
    ranking = sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)
    print(f"\n=== Query: {query!r} ===")
    for title, score in ranking:
        print(f"  {title:<25s} {score:.3f}")

In [5]:
queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet",
]
for q in queries:
    rank_documents(q)


=== Query: 'renewable energy and climate policy' ===
  Climate Change            0.323
  Economics Inflation       0.095
  Gardening Vegetables      0.048
  AI in Healthcare          0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Cybersecurity             0.000
  Music Theory              0.000
  Public Health             0.000

=== Query: 'password security and phishing' ===
  Cybersecurity             0.176
  AI in Healthcare          0.000
  Climate Change            0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Music Theory              0.000
  Gardening Vegetables      0.000
  Economics Inflation       0.000
  Public Health             0.000

=== Query: 'healthy sleep exercise and diet' ===
  Public Health             0.285
  Gardening Vegetables      0.045
  AI in Healthcare          0.000
  Climate Change            0.000
  Basketball Strategy       0.0

## Challenge

Try the queries below and explain the rankings:
- "password security and phishing"
- "healthy sleep exercise and diet"

Then re-run the vectorizer with:
    TfidfVectorizer(stop_words="english",
                    ngram_range=(1, 2),
                    max_df=0.8,
                    min_df=2)
How does the vocabulary size change? Do the top terms become more meaningful?

In [6]:
vectorizer_v2 = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2,
)
X_v2 = vectorizer_v2.fit_transform(documents)
print("v2 Matrix shape:", X_v2.shape)
print("v2 Vocabulary size:", len(vectorizer_v2.get_feature_names_out()))

v2 Matrix shape: (10, 69)
v2 Vocabulary size: 69


## Exit ticket
1. What does a high TF-IDF score mean for a term in one document?
2. Why does IDF reduce the weight of common words like "the" or "data"?
3. Name one real task where TF-IDF is useful.

In [7]:
print("Activity complete.")

Activity complete.
